In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import sys

import cv2

import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F

from box import Box
import yaml

import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation as R
import random 

import time 

root_dir = os.path.abspath('../..')
if root_dir not in sys.path:
    sys.path.append(root_dir)

from src.models.update import Update, neighbours
from src.models.graph_inference import Graph as Graph_interference
from src.models.graph_training import Graph as Graph_train
from src.models.bundle_adjustment import BundleAdjustment
from src.models.dpso import DPSO 

OSError: Could not load this library: C:\Users\janis\miniconda3\envs\def\Lib\site-packages\torch_scatter\_version_cpu.pyd

In [ ]:
model_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/model.yaml'
sonar_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/sonar.yaml'
train_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/training.yaml'

# with open(model_config_pth, "r") as f:
#     model_config = Box(yaml.safe_load(f))

# with open(sonar_config_pth, "r") as f:
#     sonar_config = Box(yaml.safe_load(f))

# with open(train_config_pth, "r") as f:
#     train_config = Box(yaml.safe_load(f))


# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# init graph instance
# PatchGraph_i = Graph_interference(model_config, sonar_config)
# PatchGraph_t = Graph_train(model_config, sonar_config, train_config)

# # init update operater instance
# UpdateOperator_t = Update(model_config)
# UpdateOperator_i = Update(model_config)



In [ ]:
# test data 
start_num = 200
data_pth_root = f'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/seq_1/fls' # 200.png'

tmp_img = cv2.imread(os.path.join(data_pth_root, f'0.png'), 0)
h, w = tmp_img.shape

frames_in_series = 5
batch_size = 3
t_0 = 1.0
dt = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

time_tensor = torch.tensor([t_0 + i*dt for i in range(frames_in_series)], device = device, dtype = torch.float)
time_tensor = time_tensor.unsqueeze(0)
time_tensor = torch.cat([time_tensor for _ in range(batch_size)], dim = 0)

# read frame 

frame_sim_b = torch.zeros((batch_size, frames_in_series, 1, h, w))

for j in range(batch_size):
    for i in range(frames_in_series):
        pth = os.path.join(data_pth_root, f'{start_num+batch_size*j+i}.png')
        frame_sim = cv2.imread(pth, 0)
        frame_sim = frame_sim.astype(np.uint8)
        
        frame_sim_b[j, i, :, :] = torch.from_numpy(frame_sim).unsqueeze(0).float()


print('-'*80)
print(f'Input data format:')
print(f'simulated tensor shape: {frame_sim.shape}, data type: {frame_sim.dtype}')
print(f'simulated tensors batch shape: {frame_sim_b.shape}, data type: {frame_sim_b.dtype}')
print(f'time tensor: {time_tensor.shape}')
print('-'*80)

--------------------------------------------------------------------------------
Input data format:
simulated tensor shape: (800, 768), data type: uint8
simulated tensors batch shape: torch.Size([3, 5, 1, 800, 768]), data type: torch.float32
time tensor: torch.Size([3, 5])
--------------------------------------------------------------------------------


# **Training test**

In [ ]:
dpso_t = DPSO(model_config_pth, sonar_config_pth, train_config_pth, mode = 'train', output_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/test')

## forward:

In [ ]:
# output = dpso_t(frame_sim_b, time_tensor)

In [ ]:
# print(output.shape)


## backward:

In [ ]:
optimizer = torch.optim.Adam(dpso_t.parameters(), lr=1e-4)
optimizer.zero_grad()

pred = dpso_t(frame_sim_b, time_tensor)

gt = torch.zeros_like(pred.clone())

criterion = nn.MSELoss()
loss = criterion(pred, gt)

print(f'loss: {loss}')

loss.backward()

optimizer.step()

In [ ]:
print(pred)

# **Inference test**

In [ ]:
dpso_i = DPSO(model_config_pth, sonar_config_pth, train_config_pth, mode = 'inference', output_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/test')

dpso_i.eval()

DPSO(
  (PatchGraph): Graph(
    (patchifier): Patchifier(
      (feature_extractor): Encoder(
        (conv1): Conv2d(1, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
        (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (ResBlock1): Sequential(
          (0): ResidualBlock(
            (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
            (norm2): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
          )
          (1): ResidualBlock(
            (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (conv2): Conv2d(32, 32, kernel_size=(3, 3), str

## forward:

In [ ]:
b, n, c, h, w = frame_sim_b.shape
print(f'Batch size: {b}, frames per batch: {n}')

prev_t = 0
# with torch.inference_mode(): # DO NOT USE THIS! this mode doesn't allow to turn grad on and it kills bundle adjustment 
with torch.no_grad():
    for batch in range(b):
        for frame_num in range(n):


            frame = frame_sim_b[batch, frame_num, :, :, :].unsqueeze(0).unsqueeze(0)
            t = time_tensor[batch, frame_num] + prev_t


            pos, time, nu = dpso_i(frame, t)
            print(f't: {time}, n: {nu}, pos: {pos}')
        prev_t = t

    dpso_i.close()

Batch size: 3, frames per batch: 5
t: 1.0, n: 1, pos: tensor([0., 0., 0., 0., 0., 0., 1.])
t: 1.5, n: 2, pos: tensor([-8.1992e-03,  7.8948e-02,  1.4508e-02,  2.3819e-03, -1.6401e-01,
         6.5894e-04,  9.8646e-01])
t: 2.0, n: 3, pos: tensor([-8.1992e-03,  7.8948e-02,  1.4508e-02,  2.3819e-03, -1.6401e-01,
         6.5894e-04,  9.8646e-01])
t: 2.5, n: 4, pos: tensor([-8.1992e-03,  7.8948e-02,  1.4508e-02,  2.3819e-03, -1.6401e-01,
         6.5894e-04,  9.8646e-01])
t: 3.0, n: 5, pos: tensor([-8.1992e-03,  7.8948e-02,  1.4508e-02,  2.3819e-03, -1.6401e-01,
         6.5894e-04,  9.8646e-01])
t: 4.0, n: 6, pos: tensor([-0.1918, -1.1627, -0.0322,  0.0838,  0.4161, -0.0968,  0.9003])
t: 4.5, n: 7, pos: tensor([ 0.6414, -1.9252,  0.3403,  0.1843,  0.6408,  0.1278,  0.7342])
t: 5.0, n: 8, pos: tensor([-0.1486, -2.6882, -0.3572,  0.1196,  0.1432,  0.4147,  0.8906])
t: 5.5, n: 9, pos: tensor([-1.5269, -4.3140,  0.1766, -0.0761,  0.0641,  0.7348,  0.6710])
t: 6.0, n: 10, pos: tensor([-1.6993, 

In [ ]:
ckpt_pth = os.path.join('C:/Users/janis/Projekty/Magisterka/SonarOdometry/test', 'test_model.cpkt')
torch.save(dpso_t.state_dict(), ckpt_pth)

model_inference = DPSO(model_config_pth, sonar_config_pth, train_config_pth, mode = 'inference', output_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/test')

model_inference.load_state_dict(torch.load(ckpt_pth, weights_only=False))
model_inference.eval()

DPSO(
  (PatchGraph): Graph(
    (patchifier): Patchifier(
      (feature_extractor): Encoder(
        (conv1): Conv2d(1, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
        (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (ResBlock1): Sequential(
          (0): ResidualBlock(
            (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
            (norm2): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
          )
          (1): ResidualBlock(
            (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (conv2): Conv2d(32, 32, kernel_size=(3, 3), str